In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, make_scorer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR 

from sklearn.model_selection import cross_val_predict
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("../data/StudentPerformance.csv")
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [ ]:
df["Extracurricular Activities"] = df["Extracurricular Activities"].map({
    "Yes": 1,
    "No": 0
})

In [ ]:
X = df.drop("Performance Index", axis=1)
y = df["Performance Index"]

In [ ]:
df.isna().sum()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scorer = make_scorer(root_mean_squared_error)

Modelo 1 - Regressão Linear

In [ ]:
linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

r2_linear = cross_val_score(
    linear_pipeline, X, y, cv=kf, scoring="r2"
).mean()

rmse_linear = cross_val_score(
    linear_pipeline, X, y, cv=kf, scoring=rmse_scorer
).mean()

r2_linear, rmse_linear


Modelo 2 - SRV

In [ ]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR())
])

svr_param_grid = {
    "model__kernel": ["rbf", "linear"],
    "model__C": [0.1, 1, 10],
    "model__epsilon": [0.1, 0.2]
}

svr_grid = GridSearchCV(
    svr_pipeline,
    svr_param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

svr_grid.fit(X, y)


Modelo 3 - Ridge Regression

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

ridge_param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

ridge_grid.fit(X, y)

In [ ]:
def avaliar_modelo(modelo):
    r2 = cross_val_score(
        modelo, X, y, cv=kf, scoring="r2"
    ).mean()
    
    rmse = cross_val_score(
        modelo, X, y, cv=kf, scoring=rmse_scorer
    ).mean()
    
    return r2, rmse

In [ ]:
resultados = []

# Regressão Linear
resultados.append([
    "Regressão Linear",
    "-",
    r2_linear,
    rmse_linear
])

# SVR
r2, rmse = avaliar_modelo(svr_grid.best_estimator_)
resultados.append([
    "SVR",
    svr_grid.best_params_,
    r2,
    rmse
])

# Ridge
r2, rmse = avaliar_modelo(ridge_grid.best_estimator_)
resultados.append([
    "Ridge Regression",
    ridge_grid.best_params_,
    r2,
    rmse
])


In [ ]:
resultados_df = pd.DataFrame(
    resultados,
    columns=[
        "Modelo",
        "Melhores Hiperparâmetros",
        "R² Médio",
        "RMSE Médio"
    ]
)

resultados_df


In [ ]:
y_pred_linear = cross_val_predict(
    linear_pipeline, X, y, cv=kf
)

residuos_linear = y - y_pred_linear

plt.style.use("seaborn-v0_8-pastel")
plt.figure()
plt.scatter(y_pred_linear, residuos_linear, alpha=0.6, s=30, edgecolors="white", linewidths=0.5)
plt.axhline(0)
plt.xlabel("Valores Preditos")
plt.ylabel("Resíduos")
plt.title("Gráfico de Resíduos - Regressão Linear")
plt.show()

In [ ]:
y_pred_svr = cross_val_predict(
    svr_grid.best_estimator_, X, y, cv=kf
)

residuos_svr = y - y_pred_svr

plt.style.use("seaborn-v0_8-pastel")
plt.figure()
plt.scatter(y_pred_linear, residuos_linear, alpha=0.6, s=30, edgecolors="white", linewidths=0.5)
plt.axhline(0)
plt.xlabel("Valores Preditos")
plt.ylabel("Resíduos")
plt.title("Gráfico de Resíduos - SVR")
plt.show()


In [ ]:
y_pred_ridge = cross_val_predict(
    ridge_grid.best_estimator_, X, y, cv=kf
)

residuos_ridge = y - y_pred_ridge

plt.style.use("seaborn-v0_8-pastel")
plt.figure()
plt.scatter(y_pred_linear, residuos_linear, alpha=0.6, s=30, edgecolors="white", linewidths=0.5)
plt.axhline(0)
plt.xlabel("Valores Preditos")
plt.ylabel("Resíduos")
plt.title("Gráfico de Resíduos - Ridge Regression")
plt.show()
